# 07 — Supervised Machine Learning


In [1]:
#Imports

from pathlib import Path
import json
import time
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    StratifiedKFold,
    KFold,
    cross_validate,
)

from sklearn.linear_model import (
    LogisticRegression,
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet,
)

from sklearn.tree import (
    DecisionTreeClassifier,
    DecisionTreeRegressor,
)

from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor,
    ExtraTreesClassifier,
    ExtraTreesRegressor,
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
)

from sklearn.svm import SVC, SVR
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier, MLPRegressor

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

ROOT = Path.cwd().parent
FEATURE_DIR = ROOT / "artifacts" / "features"
MODEL_DIR = ROOT / "artifacts" / "models"
PREDICTION_DIR = ROOT / "artifacts" / "predictions"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
#Loading Artifacts

X_train = pd.read_parquet(
    FEATURE_DIR / "X_train_raw.parquet"
)

X_test = pd.read_parquet(
    FEATURE_DIR / "X_test_raw.parquet"
)

with open(
    FEATURE_DIR / "feature_metadata.json",
    "r",
    encoding="utf-8",
) as file:
    feature_metadata = json.load(file)

TARGET_COLUMN = feature_metadata["target_column"]
problem_type = feature_metadata["problem_type"]

y_train = pd.read_parquet(
    FEATURE_DIR / "y_train.parquet"
)[TARGET_COLUMN]

y_test = pd.read_parquet(
    FEATURE_DIR / "y_test.parquet"
)[TARGET_COLUMN]

saved_preprocessor = joblib.load(
    FEATURE_DIR / "preprocessor.joblib"
)

print("Problem type:", problem_type)
print("Target:", TARGET_COLUMN)
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)


Problem type: classification
Target: purchased
Training features: (400, 5)
Testing features: (100, 5)
Training target: (400,)
Testing target: (100,)


In [ ]:
#Model catalogue

X_train = pd.read_parquet(
    FEATURE_DIR / "X_train_raw.parquet"
)

X_test = pd.read_parquet(
    FEATURE_DIR / "X_test_raw.parquet"
)

with open(
    FEATURE_DIR / "feature_metadata.json",
    "r",
    encoding="utf-8",
) as file:
    feature_metadata = json.load(file)

TARGET_COLUMN = feature_metadata["target_column"]
problem_type = feature_metadata["problem_type"]

y_train = pd.read_parquet(
    FEATURE_DIR / "y_train.parquet"
)[TARGET_COLUMN]

y_test = pd.read_parquet(
    FEATURE_DIR / "y_test.parquet"
)[TARGET_COLUMN]

saved_preprocessor = joblib.load(
    FEATURE_DIR / "preprocessor.joblib"
)

print("Problem type:", problem_type)
print("Target:", TARGET_COLUMN)
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Problem type: classification
Target: purchased
Training features: (400, 5)
Testing features: (100, 5)
Training target: (400,)
Testing target: (100,)


In [ ]:
#Optional models

def build_model_catalogue(problem_type):
    models = {}

    if problem_type == "classification":
        models = {
            "Logistic Regression": LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
            "Decision Tree": DecisionTreeClassifier(
                max_depth=8,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
            "Random Forest": RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced",
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
            "Extra Trees": ExtraTreesClassifier(
                n_estimators=300,
                class_weight="balanced",
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
            "Gradient Boosting": GradientBoostingClassifier(
                n_estimators=200,
                random_state=RANDOM_STATE,
            ),
            "Histogram Gradient Boosting":
                HistGradientBoostingClassifier(
                    max_iter=200,
                    random_state=RANDOM_STATE,
                ),
            "Support Vector Machine": SVC(
                probability=True,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
            "K-Nearest Neighbours": KNeighborsClassifier(
                n_neighbors=7,
            ),
            "Gaussian Naive Bayes": GaussianNB(),
            "ANN MLP": MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation="relu",
                solver="adam",
                early_stopping=True,
                validation_fraction=0.15,
                max_iter=500,
                random_state=RANDOM_STATE,
            ),
        }

    else:
        models = {
            "Linear Regression": LinearRegression(),
            "Ridge Regression": Ridge(alpha=1.0),
            "Lasso Regression": Lasso(
                alpha=0.01,
                max_iter=5000,
                random_state=RANDOM_STATE,
            ),
            "Elastic Net": ElasticNet(
                alpha=0.01,
                l1_ratio=0.5,
                max_iter=5000,
                random_state=RANDOM_STATE,
            ),
            "Decision Tree": DecisionTreeRegressor(
                max_depth=8,
                random_state=RANDOM_STATE,
            ),
            "Random Forest": RandomForestRegressor(
                n_estimators=300,
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
            "Extra Trees": ExtraTreesRegressor(
                n_estimators=300,
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
            "Gradient Boosting": GradientBoostingRegressor(
                n_estimators=200,
                random_state=RANDOM_STATE,
            ),
            "Histogram Gradient Boosting":
                HistGradientBoostingRegressor(
                    max_iter=200,
                    random_state=RANDOM_STATE,
                ),
            "Support Vector Regression": SVR(),
            "K-Nearest Neighbours": KNeighborsRegressor(
                n_neighbors=7,
            ),
            "ANN MLP": MLPRegressor(
                hidden_layer_sizes=(128, 64),
                activation="relu",
                solver="adam",
                early_stopping=True,
                validation_fraction=0.15,
                max_iter=500,
                random_state=RANDOM_STATE,
            ),
        }

    return models


model_catalogue = build_model_catalogue(problem_type)

print("Base models:")

for model_name in model_catalogue:
    print(model_name)

Base models:
Logistic Regression
Decision Tree
Random Forest
Extra Trees
Gradient Boosting
Histogram Gradient Boosting
Support Vector Machine
K-Nearest Neighbours
Gaussian Naive Bayes
ANN MLP


In [5]:
#Choosing models

SELECTED_MODELS = [
    "Logistic Regression",
    "Decision Tree",
    "Random Forest",
    "Extra Trees",
    "Gradient Boosting",
    "Histogram Gradient Boosting",
    "Support Vector Machine",
    "K-Nearest Neighbours",
    "Gaussian Naive Bayes",
    "ANN MLP",
]

for optional_model in [
    "XGBoost",
    "LightGBM",
    "CatBoost",
]:
    if optional_model in model_catalogue:
        SELECTED_MODELS.append(optional_model)

print("Selected models:")

for model_name in SELECTED_MODELS:
    print(model_name)

Selected models:
Logistic Regression
Decision Tree
Random Forest
Extra Trees
Gradient Boosting
Histogram Gradient Boosting
Support Vector Machine
K-Nearest Neighbours
Gaussian Naive Bayes
ANN MLP


In [6]:
#Cross validation Configuration

if problem_type == "classification":
    cross_validator = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    scoring = {
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy",
        "precision_macro": "precision_macro",
        "recall_macro": "recall_macro",
        "f1_macro": "f1_macro",
        "f1_weighted": "f1_weighted",
    }

else:
    cross_validator = KFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    scoring = {
        "mae": "neg_mean_absolute_error",
        "mse": "neg_mean_squared_error",
        "r2": "r2",
    }

print("Cross-validation folds:", 5)
print("Scoring metrics:", list(scoring))

Cross-validation folds: 5
Scoring metrics: ['accuracy', 'balanced_accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'f1_weighted']


In [7]:
#Run Cross validation

cross_validation_results = []
failed_models = {}

for model_name in SELECTED_MODELS:
    print(f"Evaluating: {model_name}")

    model = model_catalogue[model_name]

    model_pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                clone(saved_preprocessor),
            ),
            (
                "model",
                clone(model),
            ),
        ]
    )

    start_time = time.time()

    try:
        scores = cross_validate(
            estimator=model_pipeline,
            X=X_train,
            y=y_train,
            cv=cross_validator,
            scoring=scoring,
            n_jobs=-1,
            return_train_score=True,
            error_score="raise",
        )

        elapsed_seconds = time.time() - start_time

        result = {
            "model": model_name,
            "training_seconds": elapsed_seconds,
        }

        if problem_type == "classification":
            result.update({
                "cv_accuracy_mean":
                    scores["test_accuracy"].mean(),
                "cv_accuracy_std":
                    scores["test_accuracy"].std(),
                "cv_balanced_accuracy_mean":
                    scores["test_balanced_accuracy"].mean(),
                "cv_precision_macro_mean":
                    scores["test_precision_macro"].mean(),
                "cv_recall_macro_mean":
                    scores["test_recall_macro"].mean(),
                "cv_f1_macro_mean":
                    scores["test_f1_macro"].mean(),
                "cv_f1_macro_std":
                    scores["test_f1_macro"].std(),
                "cv_f1_weighted_mean":
                    scores["test_f1_weighted"].mean(),
                "train_f1_macro_mean":
                    scores["train_f1_macro"].mean(),
            })

        else:
            result.update({
                "cv_mae_mean":
                    -scores["test_mae"].mean(),
                "cv_mae_std":
                    scores["test_mae"].std(),
                "cv_rmse_mean":
                    np.sqrt(-scores["test_mse"].mean()),
                "cv_r2_mean":
                    scores["test_r2"].mean(),
                "cv_r2_std":
                    scores["test_r2"].std(),
                "train_r2_mean":
                    scores["train_r2"].mean(),
            })

        cross_validation_results.append(result)

    except Exception as error:
        failed_models[model_name] = str(error)
        print(f"Failed: {error}")

Evaluating: Logistic Regression
Evaluating: Decision Tree
Evaluating: Random Forest
Evaluating: Extra Trees
Evaluating: Gradient Boosting
Evaluating: Histogram Gradient Boosting
Evaluating: Support Vector Machine
Evaluating: K-Nearest Neighbours
Evaluating: Gaussian Naive Bayes
Evaluating: ANN MLP


In [8]:
#Cross-validation results

cv_results_df = pd.DataFrame(
    cross_validation_results
)

if cv_results_df.empty:
    raise RuntimeError(
        "All selected models failed during cross-validation."
    )

if problem_type == "classification":
    cv_results_df = cv_results_df.sort_values(
        by=[
            "cv_f1_macro_mean",
            "cv_balanced_accuracy_mean",
        ],
        ascending=False,
    )

else:
    cv_results_df = cv_results_df.sort_values(
        by="cv_r2_mean",
        ascending=False,
    )

cv_results_df = cv_results_df.reset_index(drop=True)

display(cv_results_df.round(4))

if failed_models:
    print("\nFailed models:")

    for model_name, error in failed_models.items():
        print(f"{model_name}: {error}")
        

,model,training_seconds,cv_accuracy_mean,cv_accuracy_std,cv_balanced_accuracy_mean,cv_precision_macro_mean,cv_recall_macro_mean,cv_f1_macro_mean,cv_f1_macro_std,cv_f1_weighted_mean,train_f1_macro_mean
0,Decision Tree,1.7159,0.8300,0.0302,0.8285,0.8221,0.8285,0.8238,0.0316,0.8308,0.9676
1,Gradient Boosting,2.2444,0.8275,0.0166,0.8267,0.8189,0.8267,0.8214,0.0179,0.8285,1.0000
2,Random Forest,2.3260,0.8250,0.0194,0.8248,0.8170,0.8248,0.8189,0.0214,0.8259,1.0000
3,Histogram Gradient Boosting,2.0147,0.8025,0.0267,0.7939,0.7947,0.7939,0.7928,0.0291,0.8023,1.0000
4,Logistic Regression,2.9746,0.7775,0.0470,0.7763,0.7692,0.7763,0.7708,0.0474,0.7791,0.7795
5,Support Vector Machine,1.9758,0.7650,0.0348,0.7773,0.7651,0.7773,0.7620,0.0349,0.7676,0.8350
6,K-Nearest Neighbours,0.1424,0.7675,0.0322,0.7517,0.7584,0.7517,0.7523,0.0398,0.7651,0.8235
7,Extra Trees,2.6168,0.7625,0.0403,0.7442,0.7528,0.7442,0.7463,0.0461,0.7599,1.0000
8,ANN MLP,0.1938,0.7525,0.0339,0.7241,0.7466,0.7241,0.7290,0.0414,0.7460,0.7759
9,Gaussian Naive Bayes,0.0795,0.7425,0.0281,0.7125,0.7365,0.7125,0.7184,0.0288,0.7361,0.7652


In [9]:
#Model fit and evaluation

fitted_models = {}
test_results = []
test_predictions = {}

for model_name in cv_results_df["model"]:
    print(f"Training final model: {model_name}")

    model_pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                clone(saved_preprocessor),
            ),
            (
                "model",
                clone(model_catalogue[model_name]),
            ),
        ]
    )

    try:
        model_pipeline.fit(X_train, y_train)

        predictions = model_pipeline.predict(X_test)

        fitted_models[model_name] = model_pipeline
        test_predictions[model_name] = predictions

        if problem_type == "classification":
            result = {
                "model": model_name,
                "test_accuracy": accuracy_score(
                    y_test,
                    predictions,
                ),
                "test_balanced_accuracy":
                    balanced_accuracy_score(
                        y_test,
                        predictions,
                    ),
                "test_precision_macro":
                    precision_score(
                        y_test,
                        predictions,
                        average="macro",
                        zero_division=0,
                    ),
                "test_recall_macro":
                    recall_score(
                        y_test,
                        predictions,
                        average="macro",
                        zero_division=0,
                    ),
                "test_f1_macro":
                    f1_score(
                        y_test,
                        predictions,
                        average="macro",
                        zero_division=0,
                    ),
                "test_f1_weighted":
                    f1_score(
                        y_test,
                        predictions,
                        average="weighted",
                        zero_division=0,
                    ),
            }

            if (
                y_test.nunique() == 2
                and hasattr(model_pipeline, "predict_proba")
            ):
                probabilities = model_pipeline.predict_proba(
                    X_test
                )[:, 1]

                result["test_roc_auc"] = roc_auc_score(
                    y_test,
                    probabilities,
                )

        else:
            result = {
                "model": model_name,
                "test_mae": mean_absolute_error(
                    y_test,
                    predictions,
                ),
                "test_rmse": np.sqrt(
                    mean_squared_error(
                        y_test,
                        predictions,
                    )
                ),
                "test_r2": r2_score(
                    y_test,
                    predictions,
                ),
            }

        test_results.append(result)

    except Exception as error:
        print(f"Final training failed: {error}")

Training final model: Decision Tree
Training final model: Gradient Boosting
Training final model: Random Forest
Training final model: Histogram Gradient Boosting
Training final model: Logistic Regression
Training final model: Support Vector Machine
Training final model: K-Nearest Neighbours
Training final model: Extra Trees
Training final model: ANN MLP
Training final model: Gaussian Naive Bayes


In [10]:
#Test leader board

test_results_df = pd.DataFrame(test_results)

if problem_type == "classification":
    test_results_df = test_results_df.sort_values(
        by=[
            "test_f1_macro",
            "test_balanced_accuracy",
        ],
        ascending=False,
    )
else:
    test_results_df = test_results_df.sort_values(
        by="test_r2",
        ascending=False,
    )

test_results_df = test_results_df.reset_index(drop=True)

display(test_results_df.round(4))

,model,test_accuracy,test_balanced_accuracy,test_precision_macro,test_recall_macro,test_f1_macro,test_f1_weighted,test_roc_auc
0,Random Forest,0.90,0.9134,0.8948,0.9134,0.8980,0.9011,0.9361
1,Histogram Gradient Boosting,0.86,0.8714,0.8547,0.8714,0.8572,0.8616,0.9323
2,K-Nearest Neighbours,0.84,0.8411,0.8311,0.8411,0.8346,0.8412,0.9142
3,Gaussian Naive Bayes,0.84,0.8365,0.8309,0.8365,0.8333,0.8407,0.9134
4,Gradient Boosting,0.83,0.8375,0.8233,0.8375,0.8261,0.8318,0.9361
5,ANN MLP,0.83,0.8329,0.8214,0.8329,0.8249,0.8315,0.9092
6,Support Vector Machine,0.82,0.8386,0.8223,0.8386,0.8182,0.8222,0.8949
7,Logistic Regression,0.82,0.8293,0.8145,0.8293,0.8164,0.8221,0.9071
8,Extra Trees,0.82,0.8062,0.8125,0.8062,0.8090,0.8191,0.9151
9,Decision Tree,0.80,0.7991,0.7903,0.7991,0.7933,0.8015,0.8361


In [11]:
#Best model selection

BEST_MODEL_NAME = cv_results_df.iloc[0]["model"]

best_model = fitted_models[BEST_MODEL_NAME]
best_predictions = test_predictions[BEST_MODEL_NAME]

print("Best model:", BEST_MODEL_NAME)

Best model: Decision Tree


In [12]:
#Classification evaluation

if problem_type == "classification":
    print("Classification report:")
    
    print(
        classification_report(
            y_test,
            best_predictions,
            zero_division=0,
        )
    )

    confusion_matrix_values = confusion_matrix(
        y_test,
        best_predictions,
    )

    confusion_matrix_df = pd.DataFrame(
        confusion_matrix_values,
        index=[
            f"Actual {value}"
            for value in sorted(y_test.unique())
        ],
        columns=[
            f"Predicted {value}"
            for value in sorted(y_test.unique())
        ],
    )

    display(confusion_matrix_df)

Classification report:
              precision    recall  f1-score   support

           0       0.72      0.79      0.76        39
           1       0.86      0.80      0.83        61

    accuracy                           0.80       100
   macro avg       0.79      0.80      0.79       100
weighted avg       0.81      0.80      0.80       100



,Predicted 0,Predicted 1
Actual 0,31,8
Actual 1,12,49


In [13]:
# Comparison(Training & Validation)

if problem_type == "classification":
    overfitting_df = cv_results_df[
        [
            "model",
            "train_f1_macro_mean",
            "cv_f1_macro_mean",
        ]
    ].copy()

    overfitting_df["train_validation_gap"] = (
        overfitting_df["train_f1_macro_mean"]
        - overfitting_df["cv_f1_macro_mean"]
    )

else:
    overfitting_df = cv_results_df[
        [
            "model",
            "train_r2_mean",
            "cv_r2_mean",
        ]
    ].copy()

    overfitting_df["train_validation_gap"] = (
        overfitting_df["train_r2_mean"]
        - overfitting_df["cv_r2_mean"]
    )

display(overfitting_df.round(4))

,model,train_f1_macro_mean,cv_f1_macro_mean,train_validation_gap
0,Decision Tree,0.9676,0.8238,0.1438
1,Gradient Boosting,1.0000,0.8214,0.1786
2,Random Forest,1.0000,0.8189,0.1811
3,Histogram Gradient Boosting,1.0000,0.7928,0.2072
4,Logistic Regression,0.7795,0.7708,0.0086
5,Support Vector Machine,0.8350,0.7620,0.0730
6,K-Nearest Neighbours,0.8235,0.7523,0.0712
7,Extra Trees,1.0000,0.7463,0.2537
8,ANN MLP,0.7759,0.7290,0.0469
9,Gaussian Naive Bayes,0.7652,0.7184,0.0468


In [14]:
#Save predictions

prediction_output = X_test.reset_index(drop=True).copy()

prediction_output["actual"] = (
    y_test.reset_index(drop=True)
)

prediction_output["prediction"] = best_predictions

if (
    problem_type == "classification"
    and y_test.nunique() == 2
    and hasattr(best_model, "predict_proba")
):
    prediction_output["probability_class_1"] = (
        best_model.predict_proba(X_test)[:, 1]
    )

prediction_output.to_csv(
    PREDICTION_DIR / "supervised_test_predictions.csv",
    index=False,
)

display(prediction_output.head(10))

,age,income,region,visits,satisfaction,actual,prediction,probability_class_1
0,44,24497,South,11,4.1,1,1,1.000000
1,33,68313,East,12,2.1,1,1,1.000000
2,58,97385,South,9,3.6,1,1,1.000000
3,65,102494,East,1,1.4,0,1,1.000000
4,33,31307,South,8,1.2,0,0,0.000000
5,54,73768,South,16,1.9,1,1,1.000000
6,52,95009,East,4,1.3,0,0,0.092571
7,51,85960,South,12,1.5,1,1,1.000000
8,38,94401,West,19,4.1,1,1,1.000000
9,63,127974,South,15,3.4,1,1,1.000000


In [15]:
#Save model metrics

joblib.dump(
    best_model,
    MODEL_DIR / "best_supervised_model.joblib",
)

cv_results_df.to_csv(
    MODEL_DIR / "cross_validation_results.csv",
    index=False,
)

test_results_df.to_csv(
    MODEL_DIR / "test_results.csv",
    index=False,
)

model_summary = {
    "best_model": BEST_MODEL_NAME,
    "problem_type": problem_type,
    "target_column": TARGET_COLUMN,
    "selection_metric": (
        "cv_f1_macro_mean"
        if problem_type == "classification"
        else "cv_r2_mean"
    ),
    "training_rows": len(X_train),
    "testing_rows": len(X_test),
    "selected_models": SELECTED_MODELS,
    "failed_models": failed_models,
}

with open(
    MODEL_DIR / "model_summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        model_summary,
        file,
        indent=4,
    )

print("Best model saved successfully.")
print(
    "Model path:",
    MODEL_DIR / "best_supervised_model.joblib",
)

Best model saved successfully.
Model path: e:\Practice_PROJECTS\BI_Intelligence\artifacts\models\best_supervised_model.joblib


In [16]:
#Verification of saved outputs

required_files = [
    MODEL_DIR / "best_supervised_model.joblib",
    MODEL_DIR / "cross_validation_results.csv",
    MODEL_DIR / "test_results.csv",
    MODEL_DIR / "model_summary.json",
    PREDICTION_DIR / "supervised_test_predictions.csv",
]

for path in required_files:
    print(f"{path.name}: {path.exists()}")

best_supervised_model.joblib: True
cross_validation_results.csv: True
test_results.csv: True
model_summary.json: True
supervised_test_predictions.csv: True
